# 02 Extract mediapipe keypoint notebook


This notebook is used to preprocess the WLASL100 ASL videos by converting each raw .mp4 sign video into MediaPipe keypoint data. It extracts hand and body landmarks from a fixed number of video frames and saves them as .npy files, so the LSTM model can train on numerical movement data instead of raw videos.

## Imports and paths

In [ ]:
from pathlib import Path
import json
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")

INDEX_FILE = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100" / "wlasl100_video_index.csv"
KEYPOINT_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100" / "keypoints"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / "asl_wlasl100_labels.json"

KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_FILE.parent.mkdir(parents=True, exist_ok=True)

print("Index file exists:", INDEX_FILE.exists())
print("Keypoint output folder:", KEYPOINT_DIR)

Index file exists: True
Keypoint output folder: E:\Be_My_Ear\data\processed\ASL\WLASL100\keypoints


## Load WLASL100 video index

In [ ]:
df = pd.read_csv(INDEX_FILE)

print("Total usable videos:", len(df))
print("Total classes:", df["label_id"].nunique())

df.head()

Total usable videos: 1013
Total classes: 100


,video_id,original_class_id,gloss,video_path,label_id
0,69422,27,orange,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,67
1,10898,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,20
2,10893,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,20
3,10892,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4,20
4,10895,82,city,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4,20


## Save label map

In [ ]:
label_map = {}

for _, row in df.drop_duplicates("label_id").iterrows():
    label_id = int(row["label_id"])
    label_map[label_id] = {
        "language": "ASL",
        "gloss": row["gloss"],
        "display_text": row["gloss"],
        "original_class_id": int(row["original_class_id"])
    }

# Sort by label_id
label_map = dict(sorted(label_map.items(), key=lambda x: x[0]))

with open(LABEL_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=4)

print("Saved label map to:")
print(LABEL_MAP_FILE)

list(label_map.items())[:5]

Saved label map to:
E:\Be_My_Ear\data\label_maps\asl_wlasl100_labels.json


[(0,
  {'language': 'ASL',
   'gloss': 'accident',
   'display_text': 'accident',
   'original_class_id': 51}),
 (1,
  {'language': 'ASL',
   'gloss': 'africa',
   'display_text': 'africa',
   'original_class_id': 76}),
 (2,
  {'language': 'ASL',
   'gloss': 'all',
   'display_text': 'all',
   'original_class_id': 18}),
 (3,
  {'language': 'ASL',
   'gloss': 'apple',
   'display_text': 'apple',
   'original_class_id': 52}),
 (4,
  {'language': 'ASL',
   'gloss': 'basketball',
   'display_text': 'basketball',
   'original_class_id': 77})]

## Test if one video opens

In [ ]:
test_video = df.iloc[0]["video_path"]

cap = cv2.VideoCapture(test_video)

print("Testing video:")
print(test_video)
print("Opened:", cap.isOpened())
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

cap.release()

Testing video:
E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4
Opened: True
Frame count: 59
FPS: 29.97002997002997


## Create landmark extraction function

This gives every video the same shape:

[60, 258]

Why 258?

Left hand: 21 landmarks × 3 = 63
Right hand: 21 landmarks × 3 = 63
Pose: 33 landmarks × 4 = 132

Total = 63 + 63 + 132 = 258

In [ ]:
mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = 60

LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4
FEATURE_SIZE = LEFT_HAND_SIZE + RIGHT_HAND_SIZE + POSE_SIZE

print("Feature size:", FEATURE_SIZE)

Feature size: 258


In [ ]:
def extract_landmarks_from_results(results):
    """
    Convert MediaPipe Holistic results into one flat feature vector.
    Output shape: [258]
    """

    # Left hand: 21 landmarks x 3
    if results.left_hand_landmarks:
        left_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        left_hand = np.zeros(LEFT_HAND_SIZE, dtype=np.float32)

    # Right hand: 21 landmarks x 3
    if results.right_hand_landmarks:
        right_hand = np.array(
            [[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        right_hand = np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)

    # Pose: 33 landmarks x 4
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark],
            dtype=np.float32
        ).flatten()
    else:
        pose = np.zeros(POSE_SIZE, dtype=np.float32)

    return np.concatenate([left_hand, right_hand, pose])

## Extract one video into keypoints

In [ ]:
def extract_keypoints_from_video(video_path, sequence_length=60):
    """
    Extract fixed-length MediaPipe keypoint sequence from one video.
    Output shape: [sequence_length, 258]
    """

    video_path = str(video_path)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return None

    # Sample 60 frames evenly from the video
    frame_indices = np.linspace(0, frame_count - 1, sequence_length).astype(int)

    sequence = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()

            if not success:
                sequence.append(np.zeros(FEATURE_SIZE, dtype=np.float32))
                continue

            # Convert BGR to RGB for MediaPipe
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Improve performance
            image_rgb.flags.writeable = False
            results = holistic.process(image_rgb)

            keypoints = extract_landmarks_from_results(results)
            sequence.append(keypoints)

    cap.release()

    sequence = np.array(sequence, dtype=np.float32)

    return sequence

In [ ]:
#Test extraction on one video
sample_row = df.iloc[0]
sample_video_path = sample_row["video_path"]

sample_keypoints = extract_keypoints_from_video(sample_video_path, SEQUENCE_LENGTH)

print("Video:", sample_video_path)

if sample_keypoints is None:
    print("Failed to extract keypoints.")
else:
    print("Keypoint shape:", sample_keypoints.shape)
    print("Data type:", sample_keypoints.dtype)
    print("Min:", sample_keypoints.min())
    print("Max:", sample_keypoints.max())

e:\Be_My_Ear\.venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Video: E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4
Keypoint shape: (60, 258)
Data type: float32
Min: -1.1773801
Max: 2.0495439


##  Save one sample keypoint file


In [ ]:
sample_output_path = KEYPOINT_DIR / f"{sample_row['video_id']}.npy"

np.save(sample_output_path, sample_keypoints)

print("Saved sample keypoints to:")
print(sample_output_path)
print("File exists:", sample_output_path.exists())

Saved sample keypoints to:
E:\Be_My_Ear\data\processed\ASL\WLASL100\keypoints\69422.npy
File exists: True


## Process first 10 videos as a test

In [ ]:
test_df = df.head(10).copy()

success_count = 0
fail_count = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    video_id = row["video_id"]
    video_path = row["video_path"]

    output_path = KEYPOINT_DIR / f"{video_id}.npy"

    keypoints = extract_keypoints_from_video(video_path, SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Success:", success_count)
print("Failed:", fail_count)

100%|██████████| 10/10 [00:24<00:00,  2.43s/it]

Success: 10
Failed: 0


## Process first 10 videos as a test

In [ ]:
success_count = 0
fail_count = 0
skipped_count = 0

failed_records = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    video_id = row["video_id"]
    video_path = row["video_path"]

    output_path = KEYPOINT_DIR / f"{video_id}.npy"

    # Skip if already extracted
    if output_path.exists():
        skipped_count += 1
        continue

    keypoints = extract_keypoints_from_video(video_path, SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        failed_records.append({
            "video_id": video_id,
            "video_path": video_path,
            "reason": "video_open_or_frame_failed"
        })
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Extraction completed.")
print("Success:", success_count)
print("Skipped:", skipped_count)
print("Failed:", fail_count)

100%|██████████| 1013/1013 [45:18<00:00,  2.68s/it]

Extraction completed.
Success: 1003
Skipped: 10
Failed: 0


## Save extraction failure log


In [ ]:
FAILED_LOG_FILE = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100" / "failed_keypoint_extraction.csv"

failed_df = pd.DataFrame(failed_records)
failed_df.to_csv(FAILED_LOG_FILE, index=False)

print("Failed records saved to:")
print(FAILED_LOG_FILE)
print("Failed count:", len(failed_df))

Failed records saved to:
E:\Be_My_Ear\data\processed\ASL\WLASL100\failed_keypoint_extraction.csv
Failed count: 0


## Create final keypoint index

This index tells the training notebook which .npy files are available.

In [ ]:
records = []

for _, row in df.iterrows():
    video_id = row["video_id"]
    keypoint_path = KEYPOINT_DIR / f"{video_id}.npy"

    if keypoint_path.exists():
        records.append({
            "video_id": video_id,
            "gloss": row["gloss"],
            "label_id": int(row["label_id"]),
            "original_class_id": int(row["original_class_id"]),
            "video_path": row["video_path"],
            "keypoint_path": str(keypoint_path)
        })

keypoint_df = pd.DataFrame(records)

KEYPOINT_INDEX_FILE = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100" / "wlasl100_keypoint_index.csv"
keypoint_df.to_csv(KEYPOINT_INDEX_FILE, index=False)

print("Saved keypoint index to:")
print(KEYPOINT_INDEX_FILE)

print("Total keypoint samples:", len(keypoint_df))
print("Total classes:", keypoint_df["label_id"].nunique())

keypoint_df.head()

Saved keypoint index to:
E:\Be_My_Ear\data\processed\ASL\WLASL100\wlasl100_keypoint_index.csv
Total keypoint samples: 1013
Total classes: 100


,video_id,gloss,label_id,original_class_id,video_path,keypoint_path
0,69422,orange,67,27,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...
1,10898,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...
2,10893,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...
3,10892,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...
4,10895,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...


## Check class distribution after keypoint extraction

In [ ]:
class_counts = keypoint_df["gloss"].value_counts()

print("Classes:", len(class_counts))
print("Minimum samples per class:", class_counts.min())
print("Maximum samples per class:", class_counts.max())
print("Average samples per class:", round(class_counts.mean(), 2))

class_counts.head(20)

Classes: 100
Minimum samples per class: 5
Maximum samples per class: 16
Average samples per class: 10.13


gloss
before          16
cool            16
thin            16
drink           15
go              15
cousin          14
who             14
computer        14
help            14
tall            13
candy           13
thanksgiving    13
bed             13
accident        13
bowling         13
short           13
yes             12
basketball      12
shirt           12
dark            12
Name: count, dtype: int64